# Data propare and tokenizer

In [1]:
!pip install -q \
    transformers==4.47.0 \
    bitsandbytes==0.46.1 \
    accelerate==1.2.1 \
    peft==0.14.0 \
    trl==0.13.0 \
    datasets==3.2.0

In [2]:
import json
from datasets import Dataset, DatasetDict

DATA_DIR = "./data"

print("Loading data...")
with open(f"{DATA_DIR}/train_text_label.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(f"{DATA_DIR}/test_text_label.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "test":  Dataset.from_list(test_data)
})
print(f"Finish loading, training size: {len(dataset['train'])}, testing size: {len(dataset['test'])}")

Loading data...
Finish loading, training size: 84176, testing size: 11699


In [3]:
from transformers import AutoTokenizer

model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")

Loading tokenizer...
Tokenizer loaded.


In [4]:
print("Formatting prompts...")

def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]

    prompts     = []
    completions = []

    for instruction, input_text, output in zip(instructions, inputs, outputs):
        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )
        completion = f"{output}{tokenizer.eos_token}"
        prompts.append(prompt)
        completions.append(completion)

    return {"prompt": prompts, "completion": completions}

original_columns = dataset["train"].column_names
processed_dataset = dataset.map(format_prompts, batched=True)
processed_dataset = processed_dataset.remove_columns(original_columns)

processed_dataset["train"].to_json(f"{DATA_DIR}/train_dataset_formatted.jsonl", lines=True, force_ascii=False)
processed_dataset["test"].to_json(f"{DATA_DIR}/test_dataset_formatted.jsonl",  lines=True, force_ascii=False)
print("Formatting done and saved.")

Formatting prompts...


Map:   0%|          | 0/84176 [00:00<?, ? examples/s]

Map:   0%|          | 0/11699 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/85 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/12 [00:00<?, ?ba/s]

Formatting done and saved.


# Training

!!! mention: If finish processing data, read data in this part



In [5]:
import os
import torch
from transformers import BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

training_args = SFTConfig(
    output_dir="./models/deepseek-financial-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    logging_steps=10,
    num_train_epochs=3,
    max_steps=-1,
    fp16=False,
    bf16=True,
    max_seq_length=512,
    report_to="none"
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print("Config done.")

Config done.


In [6]:
import os
import torch
from transformers import BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

training_args = SFTConfig(
    output_dir="./models/deepseek-financial-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    logging_steps=10,
    num_train_epochs=3,
    max_steps=-1,
    fp16=False,
    bf16=True,
    max_seq_length=512,
    report_to="none"
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print("Config done.")

Config done.


In [7]:
from transformers import AutoModelForCausalLM
from peft import prepare_model_for_kbit_training

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
print("Model loaded.")

Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded.


In [8]:
from trl import SFTTrainer

print("Initializing SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["test"],
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args
)
print("Trainer ready.")

Initializing SFTTrainer...


Map:   0%|          | 0/84176 [00:00<?, ? examples/s]

Map:   0%|          | 0/11699 [00:00<?, ? examples/s]

/home/jiayuanj/.local/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a processing_class with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `processing_class.padding_side = 'right'` to your code.
  warnings.warn(


Trainer ready.


In [9]:
import os
import datetime
from transformers import TrainerCallback

# ── 1. 追加日志 Callback ────────────────────────────────────────────────────
LOG_FILE = "./training_history.log"

class AppendLogCallback(TrainerCallback):
    def __init__(self, log_file: str = LOG_FILE):
        self.log_file = log_file
        with open(self.log_file, "a", encoding="utf-8") as f:
            f.write(f"\n{'='*60}\n")
            f.write(f"Training session started: {datetime.datetime.now()}\n")
            f.write(f"{'='*60}\n")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or "loss" not in logs:
            return
        ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(self.log_file, "a", encoding="utf-8") as f:
            f.write(f"[{ts}] step={state.global_step:>6d}  loss={logs['loss']:.6f}\n")

    def on_train_end(self, args, state, control, **kwargs):
        with open(self.log_file, "a", encoding="utf-8") as f:
            f.write(f"Training session ended:   {datetime.datetime.now()}\n")

# ── 2. 注入 callback ────────────────────────────────────────────────────────
trainer.add_callback(AppendLogCallback(log_file=LOG_FILE))

# ── 3. 自动检测 checkpoint 续训 ────────────────────────────────────────────
last_checkpoint = None
ckpt_dir = training_args.output_dir
if os.path.isdir(ckpt_dir):
    checkpoints = [
        os.path.join(ckpt_dir, d)
        for d in os.listdir(ckpt_dir)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        last_checkpoint = max(checkpoints, key=os.path.getmtime)
        print(f"Resuming from checkpoint: {last_checkpoint}")
    else:
        print("No checkpoint found, training from scratch.")

trainer.train(resume_from_checkpoint=last_checkpoint)

trainer.model.save_pretrained("./models/deepseek-financial-lora-final")
print("Training complete and model saved!")
print(f"Training log appended to: {LOG_FILE}")

Resuming from checkpoint: ./models/deepseek-financial-lora/checkpoint-13800


/home/jiayuanj/.local/lib/python3.11/site-packages/transformers/trainer.py:3418: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(checkpoint, OPTIMIZER_

Step,Training Loss
13810,1.764800
13820,1.856500
13830,1.930900
13840,1.571500
13850,1.323100
13860,1.705600
13870,1.685800
13880,1.613800
13890,2.202700
13900,1.554100


/home/jiayuanj/.local/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/jiayuanj/.local/lib/python3.11/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
/home/jiayuanj/.local/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In ve

Training complete and model saved!
Training log appended to: ./training_history.log
